In [38]:
from pathlib import Path
import numpy as np
import pandas as pd
from src.data.loader import load_orders, load_order_items, load_web_traffic, load_inventory, load_sales
from src.features.build import build_daily_one_row_per_day, drop_columns
from src.utils.plot_features import build_daily, plot_by_year, plot_all_year, plot_overlay

root = Path.cwd()

orders_df = load_orders()
order_items_df = load_order_items()
web_traffic_df = load_web_traffic()
inventory_df = load_inventory()
sales_df = load_sales()

# Cấu hình năm muốn loại khỏi tập build feature. Để [] để giữ toàn bộ dữ liệu (bao gồm 2012).
DROP_YEARS = []

daily_df = build_daily_one_row_per_day(
    orders_df=orders_df,
    order_items_df=order_items_df,
    web_traffic_df=web_traffic_df,
    inventory_df=inventory_df,
    sales_df=sales_df,
    drop_years=DROP_YEARS,
 )

daily_df = drop_columns(daily_df, ["same_day_delivery_orders", "weekend_order_share", "same_day_delivery_rate", "inv_reorder_products"] )

# --- Data cleaning trước khi train ---
daily_df['date'] = pd.to_datetime(daily_df['date'], errors='coerce')
daily_df = daily_df.dropna(subset=['date'])
daily_df = daily_df.sort_values('date').drop_duplicates(subset=['date'], keep='last').reset_index(drop=True)

numeric_cols = daily_df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    daily_df[numeric_cols] = daily_df[numeric_cols].replace([np.inf, -np.inf], np.nan)

    # Giữ target, fill median cho feature để ổn định train
    target_cols = ['Revenue', 'COGS']
    feature_num_cols = [c for c in numeric_cols if c not in target_cols]
    if feature_num_cols:
        daily_df[feature_num_cols] = daily_df[feature_num_cols].fillna(daily_df[feature_num_cols].median())

    for c in target_cols:
        if c in daily_df.columns:
            daily_df[c] = daily_df[c].fillna(0.0)

print("shape:", daily_df.shape)
print("duplicated date rows:", int(daily_df["date"].duplicated().sum()))
print("date range:", daily_df["date"].min(), "->", daily_df["date"].max())

print("\ncolumns:")
for i, col in enumerate(daily_df.columns, start=1):
    print(f"{i:02d}. {col}")

shape: (3833, 484)
duplicated date rows: 0
date range: 2012-07-04 00:00:00 -> 2022-12-31 00:00:00

columns:
01. date
02. Revenue
03. COGS
04. orders_count
05. customers_count
06. returned_orders
07. cancelled_orders
08. delivered_orders
09. payment_value_total
10. installments_mean
11. shipping_fee_total
12. shipping_lead_days_mean
13. shipping_lead_days_p90
14. fulfillment_days_mean
15. payment_value_mean
16. payment_value_std
17. payment_value_median
18. customer_tenure_days_mean
19. customer_tenure_days_p90
20. weekend_orders
21. same_day_ship_orders
22. return_rate_orders
23. cancel_rate_orders
24. same_day_ship_rate
25. items_count
26. quantity_sold
27. gross_merch_value
28. discount_total
29. net_merch_value
30. list_merch_value
31. items_cogs_total
32. unique_products_sold
33. unit_price_mean
34. unit_price_std
35. promo_1_active_lines
36. promo_2_active_lines
37. stackable_promo_lines
38. refund_amount_total
39. return_quantity_total
40. rating_mean
41. promo_1_usage_count
42. 

In [39]:
# 1. Build Model + Feature Pipeline (Leakage-safe: Date -> non-target features)
from src.models.sklearn_models import SklearnRegressorConfig, SklearnRegressorWrapper
import numpy as np
import pandas as pd

def build_date_feature_frame(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    dt = pd.to_datetime(df[date_col])

    # Base calendar features
    out['year'] = dt.dt.year.astype(int)
    out['quarter'] = dt.dt.quarter.astype(int)
    out['month'] = dt.dt.month.astype(int)
    out['day'] = dt.dt.day.astype(int)
    out['day_of_week'] = dt.dt.dayofweek.astype(int)
    out['day_of_year'] = dt.dt.dayofyear.astype(int)
    out['week_of_year'] = dt.dt.isocalendar().week.astype(int)

    out['is_weekend'] = (out['day_of_week'] >= 5).astype(int)
    out['is_month_start'] = dt.dt.is_month_start.astype(int)
    out['is_month_end'] = dt.dt.is_month_end.astype(int)
    out['is_quarter_start'] = dt.dt.is_quarter_start.astype(int)
    out['is_quarter_end'] = dt.dt.is_quarter_end.astype(int)

    # Cyclical encoding for calendar periodicity
    out['dow_sin'] = np.sin(2 * np.pi * out['day_of_week'] / 7.0)
    out['dow_cos'] = np.cos(2 * np.pi * out['day_of_week'] / 7.0)
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12.0)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12.0)
    out['doy_sin'] = np.sin(2 * np.pi * out['day_of_year'] / 365.25)
    out['doy_cos'] = np.cos(2 * np.pi * out['day_of_year'] / 365.25)

    # Regime-break features
    out['is_pre_2019'] = (out['year'] <= 2018).astype(int)
    out['is_post_2019'] = (out['year'] >= 2019).astype(int)
    out['is_2019_2022_regime'] = out['year'].between(2019, 2022).astype(int)
    out['is_forecast_2023_2024'] = out['year'].between(2023, 2024).astype(int)
    out['regime_year_idx'] = np.where(out['year'] < 2019, out['year'] - 2013, out['year'] - 2019).astype(float)
    out['post2019_x_month'] = out['is_post_2019'] * out['month']
    out['post2019_x_weekend'] = out['is_post_2019'] * out['is_weekend']

    # Month-end effect flags
    days_to_month_end = (dt + pd.offsets.MonthEnd(0) - dt).dt.days.astype(int)
    out['days_to_month_end'] = days_to_month_end
    out['is_month_end_5d'] = (days_to_month_end <= 4).astype(int)

    # Odd/even year asymmetry signals
    out['is_odd_year'] = (out['year'] % 2).astype(int)
    out['is_august_odd'] = ((out['month'] == 8) & (out['is_odd_year'] == 1)).astype(int)
    out['is_q3_odd'] = (out['month'].isin([7, 8, 9]) & (out['is_odd_year'] == 1)).astype(int)
    out['month_x_odd_year'] = out['month'] * out['is_odd_year']

    # Two-year cycle encoding
    day_idx = (dt - pd.Timestamp('2012-01-01')).dt.days.astype(float)
    out['biennial_sin'] = np.sin(2 * np.pi * day_idx / 730.5)
    out['biennial_cos'] = np.cos(2 * np.pi * day_idx / 730.5)

    return out

def fit_year_level_model(train_df: pd.DataFrame, target_col: str) -> dict:
    yearly = train_df.groupby('year', as_index=False)[target_col].mean().sort_values('year')
    x = yearly['year'].to_numpy(dtype=float)
    y = yearly[target_col].to_numpy(dtype=float)

    x0 = float(x.min())
    x_rel = x - x0
    X = np.column_stack([
        np.ones_like(x_rel),
        x_rel,
        (x >= 2019).astype(float),
        (x % 2 == 1).astype(float),
        (x >= 2023).astype(float),
    ])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return {'x0': x0, 'beta': beta}

def predict_year_level(year_values, model: dict) -> np.ndarray:
    y = np.asarray(year_values, dtype=float)
    x_rel = y - float(model['x0'])
    X = np.column_stack([
        np.ones_like(x_rel),
        x_rel,
        (y >= 2019).astype(float),
        (y % 2 == 1).astype(float),
        (y >= 2023).astype(float),
    ])
    return np.maximum(1e-6, X @ model['beta'])

# Có thể đổi sang 'lightgbm' hoặc 'catboost' nếu muốn benchmark
config = SklearnRegressorConfig(
    model_type='xgboost',
    random_state=42,
    n_estimators=700,
    learning_rate=0.03,
    max_depth=8,
)

model_rev = SklearnRegressorWrapper(config)
model_ratio = SklearnRegressorWrapper(config)

In [40]:
# 2. Train Model (Revenue + Ratio)
TRAIN_START_YEAR = 2012
train = daily_df[daily_df['year'] >= TRAIN_START_YEAR].copy()

# Xây feature KHÔNG chứa target: chỉ sinh từ Date
X_train = build_date_feature_frame(train, date_col='date').astype(float)
features = X_train.columns.tolist()

# Learn yearly level directly from data (không dùng scale tay trên output)
rev_level_model = fit_year_level_model(train, target_col='Revenue')
train_rev_level = predict_year_level(train['year'].to_numpy(), rev_level_model)

# Train target cho revenue ở dạng normalized
y_train_rev_raw = train['Revenue'].astype(float).to_numpy()
y_train_rev = pd.Series(y_train_rev_raw / train_rev_level, index=train.index, dtype='float64').clip(lower=0.0)

# Target 2: ratio = COGS/Revenue để suy ra COGS từ Revenue dự đoán
train_ratio = train['COGS'].astype(float) / train['Revenue'].astype(float).replace(0, np.nan)
ratio_fallback = float(train_ratio.median()) if not np.isnan(train_ratio.median()) else 0.5
y_train_ratio = train_ratio.fillna(ratio_fallback).clip(lower=0.0, upper=2.5)

print(f'Training rows: {len(train)} (from year >= {TRAIN_START_YEAR})')
print(f'Using {len(features)} non-target features from Date')
print('Sample features:', features[:10])
print('Revenue normalized target mean:', float(y_train_rev.mean()))

print('Training Revenue Model (normalized)...')
model_rev.fit(X_train, y_train_rev)

print('Training Ratio Model (COGS/Revenue)...')
model_ratio.fit(X_train, y_train_ratio)

print('Training Done!')

Training rows: 3833 (from year >= 2012)
Using 33 non-target features from Date
Sample features: ['year', 'quarter', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'is_weekend', 'is_month_start', 'is_month_end']
Revenue normalized target mean: 1.035824304986786
Training Revenue Model (normalized)...
Training Ratio Model (COGS/Revenue)...
Training Done!


In [41]:
# 3. Train Teacher (full business features, leakage-safe)
def _is_direct_target_feature(col: str) -> bool:
    lower = col.lower()
    has_target_token = ('revenue' in lower) or ('cogs' in lower) or ('gross_margin' in lower)
    has_history_token = any(tok in lower for tok in ['_lag_', '_roll_', '_diff_', '_pct_change_', '_yoy_'])
    return has_target_token and (not has_history_token)

teacher_base_exclude = {'date', 'Revenue', 'COGS'}
teacher_candidates = [
    c for c in train.columns
    if c not in teacher_base_exclude and not _is_direct_target_feature(c)
]

X_train_teacher = (
    train[teacher_candidates]
    .select_dtypes(include=[np.number])
    .astype(float)
    .fillna(0.0)
    .replace([np.inf, -np.inf], 0.0)
)
teacher_features = X_train_teacher.columns.tolist()

teacher_config = SklearnRegressorConfig(
    model_type='xgboost',
    random_state=42,
    n_estimators=900,
    learning_rate=0.03,
    max_depth=8,
)

model_teacher_rev = SklearnRegressorWrapper(teacher_config)
model_teacher_ratio = SklearnRegressorWrapper(teacher_config)

print(f'Teacher features count: {len(teacher_features)}')
print('Teacher sample features:', teacher_features[:15])

print('Training Teacher Revenue Model...')
model_teacher_rev.fit(X_train_teacher, y_train_rev)

print('Training Teacher Ratio Model...')
model_teacher_ratio.fit(X_train_teacher, y_train_ratio)

print('Teacher training done!')

Teacher features count: 435
Teacher sample features: ['orders_count', 'customers_count', 'returned_orders', 'cancelled_orders', 'delivered_orders', 'payment_value_total', 'installments_mean', 'shipping_fee_total', 'shipping_lead_days_mean', 'shipping_lead_days_p90', 'fulfillment_days_mean', 'payment_value_mean', 'payment_value_std', 'payment_value_median', 'customer_tenure_days_mean']
Training Teacher Revenue Model...
Training Teacher Ratio Model...
Teacher training done!


In [42]:
# 4. Create soft targets from Teacher predictions
teacher_pred_rev = pd.Series(model_teacher_rev.predict(X_train_teacher), index=train.index, dtype='float64')
teacher_pred_ratio = pd.Series(model_teacher_ratio.predict(X_train_teacher), index=train.index, dtype='float64')

ALPHA_SOFT = 0.7  # blend(teacher, ground-truth)
y_soft_rev = (ALPHA_SOFT * teacher_pred_rev + (1 - ALPHA_SOFT) * y_train_rev).clip(lower=0.0)
y_soft_ratio = (ALPHA_SOFT * teacher_pred_ratio + (1 - ALPHA_SOFT) * y_train_ratio).clip(lower=0.0, upper=2.5)

print('Soft target stats:')
print('Revenue soft mean:', float(y_soft_rev.mean()))
print('Ratio soft mean:', float(y_soft_ratio.mean()))

Soft target stats:
Revenue soft mean: 1.0358260809023436
Ratio soft mean: 0.8761271093360957


In [43]:
# 5. Train Student Date-only (learn from soft targets)
X_train_student = X_train.astype(float).fillna(0.0)

student_config = SklearnRegressorConfig(
    model_type='xgboost',
    random_state=42,
    n_estimators=500,
    learning_rate=0.04,
    max_depth=6,
)

model_student_rev = SklearnRegressorWrapper(student_config)
model_student_ratio = SklearnRegressorWrapper(student_config)

print('Training Student Revenue Model (Date-only)...')
model_student_rev.fit(X_train_student, y_soft_rev)

print('Training Student Ratio Model (Date-only)...')
model_student_ratio.fit(X_train_student, y_soft_ratio)

print('Student training done!')

Training Student Revenue Model (Date-only)...
Training Student Ratio Model (Date-only)...
Student training done!


In [44]:
# 6. Predict using Student only (Date-only inference, model-based yearly level)
future_dates = pd.date_range(start='2023-01-01', end='2024-07-01', freq='D')
test = pd.DataFrame({'Date': future_dates})

# Student chỉ dùng feature từ Date
X_test = build_date_feature_frame(test, date_col='Date').reindex(columns=features).astype(float)

# Predict normalized revenue + ratio
pred_rev_norm = model_student_rev.predict(X_test)
pred_ratio = model_student_ratio.predict(X_test)

pred_rev_norm = np.maximum(0, pred_rev_norm)
pred_ratio = np.clip(pred_ratio, 0.0, 2.5)

# Recover revenue level from learned yearly-level model
test_year_level = predict_year_level(test['Date'].dt.year.to_numpy(), rev_level_model)
test['Revenue'] = pred_rev_norm * test_year_level

# COGS inferred from ratio
test['COGS'] = np.maximum(0, test['Revenue'] * pred_ratio)

test['Revenue'] = test['Revenue'].round(2)
test['COGS'] = test['COGS'].round(2)

print('Forecast mean by year (model-based level):')
print(test.groupby(test['Date'].dt.year)[['Revenue', 'COGS']].mean())

test[['Date', 'Revenue', 'COGS']].head()

Forecast mean by year (model-based level):
           Revenue          COGS
Date                            
2023  3.995586e+06  3.647330e+06
2024  4.745597e+06  4.028937e+06


,Date,Revenue,COGS
0,2023-01-01,2994618.01,2887914.63
1,2023-01-02,1816995.74,1492014.80
2,2023-01-03,1333477.88,1112775.38
3,2023-01-04,1178203.49,952399.21
4,2023-01-05,1055545.40,864270.92


In [45]:
# 4. Generate Submission
submission = test[['Date', 'Revenue', 'COGS']].copy()
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')

output_path = 'submission_baseline.csv'
submission.to_csv(output_path, index=False)

print(f'Tạo file submission thành công tại {output_path}')
submission.head()

Tạo file submission thành công tại submission_baseline.csv


,Date,Revenue,COGS
0,2023-01-01,2994618.01,2887914.63
1,2023-01-02,1816995.74,1492014.80
2,2023-01-03,1333477.88,1112775.38
3,2023-01-04,1178203.49,952399.21
4,2023-01-05,1055545.40,864270.92
